# Summary_Day13_online.ipynb  
## 사전학습 Pretrained 모델 활용 1 · 인터넷 가능 버전 · ResNet18 / VGG19-BN / MobileNetV2

이번 13강은 **사전학습 모델 Pretrained Model**을 활용하는 강의다.

지금까지는 CNN 구조를 직접 만들고 CIFAR-10을 학습했다.  
이번부터는 이미 ImageNet 같은 대규모 데이터셋으로 학습된 유명 모델을 가져와서 우리 문제에 맞게 바꿔 쓴다.

강의 흐름은 다음이다.

```text
사전학습 모델 개념
→ 파인튜닝과 전이학습 차이
→ AdaptiveAvgPool2d 구조 이해
→ CIFAR-10 데이터 준비
→ ResNet18 불러오기
→ 마지막 fc layer 교체
→ 학습과 결과 평가
→ VGG19-BN 구조 확인과 classifier[6] 교체
→ Transfer Learning 방식으로 일부 파라미터 freeze
→ MobileNetV2로 초급 이미지 분류기 만들기
→ 예측 시각화, 클래스별 성능, 모델 저장
```

이 파일은 **인터넷 가능 환경** 기준이다.  
CIFAR-10 데이터와 ImageNet 사전학습 가중치를 다운로드할 수 있는 환경에서 실행한다.

> 필기 포인트:  
> 사전학습 모델은 이미 많은 이미지를 보고 선, 모서리, 질감, 물체의 기본 패턴을 배운 모델이다.  
> 우리는 처음부터 새로 만들기보다 이 모델의 마지막 분류 부분을 우리 문제에 맞게 바꿔 쓴다.

## 1. 전체 실습 목적

이번 실습의 목적은 다음이다.

1. 사전학습 모델이 무엇인지 이해한다.
2. 파인튜닝과 전이학습을 구분한다.
3. `nn.AdaptiveAvgPool2d((1,1))`이 왜 사전학습 모델에서 중요한지 확인한다.
4. CIFAR-10 이미지를 사전학습 모델 입력 크기에 맞게 준비한다.
5. ResNet18의 마지막 `fc` layer를 CIFAR-10 class 수에 맞게 교체한다.
6. VGG19-BN의 마지막 `classifier[6]` layer를 교체한다.
7. `requires_grad=False`로 전이학습용 feature extractor freeze를 적용한다.
8. MobileNetV2의 `classifier[1]`을 교체해 초급 이미지 분류기 구조를 만든다.
9. 예측 결과, 클래스별 정확도, 모델 저장 흐름을 정리한다.

## 2. 라이브러리 준비

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
```

- `torch`: Tensor 연산과 GPU 사용에 필요하다.
- `nn`: `Linear`, `CrossEntropyLoss`, `AdaptiveAvgPool2d` 등을 사용할 때 필요하다.
- `optim`: SGD, Adam 같은 Optimizer를 사용할 때 필요하다.
- `torchvision.datasets`: CIFAR-10 같은 이미지 데이터셋을 불러온다.
- `torchvision.transforms`: Resize, Normalize, RandomHorizontalFlip 같은 전처리를 구성한다.
- `torchvision.models`: ResNet, VGG, MobileNet 같은 사전학습 모델을 불러온다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(123)
np.random.seed(123)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

## 3. device와 난수 고정 함수

사전학습 모델은 크기가 커서 GPU가 있으면 GPU를 쓰는 것이 좋다.

### 함수 사용법

```python
torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
tensor.to(device)
```

- `device`: CPU 또는 GPU 위치다.
- 모델과 데이터가 서로 다른 device에 있으면 연산 에러가 난다.

```python
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
```

- 난수를 고정해 실험 재현성을 높인다.

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def torch_seed(seed=123):
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

torch_seed()

print("device:", device)

## 4. 사전학습 모델이란 무엇인가

사전학습 모델은 ImageNet 같은 대규모 데이터셋으로 미리 학습된 모델이다.

이미 알고 있는 것은 다음과 같다.

```text
선
모서리
색상 패턴
질감
단순한 모양
복잡한 물체 특징
```

우리는 이 모델을 가져와 마지막 분류 부분만 새 문제에 맞게 바꾸거나, 전체를 조금 더 학습시킨다.

```text
처음부터 학습: 랜덤한 가중치에서 시작
사전학습 활용: 이미 배운 가중치에서 시작
```

> 강의식 이해:  
> 자동차를 처음부터 만드는 것이 아니라, 이미 만들어진 자동차를 내 목적에 맞게 튜닝하는 느낌이다.

## 5. 파인튜닝과 전이학습 차이

두 용어가 헷갈리기 쉽다.

| 구분 | 의미 | 언제 적합한가 |
|---|---|---|
| Fine-Tuning | 사전학습 모델 전체 또는 많은 부분을 새 데이터로 다시 미세 조정한다 | 데이터가 비교적 많을 때 |
| Transfer Learning | 앞쪽 feature extractor는 고정하고 마지막 분류기 위주로 학습한다 | 데이터가 적을 때 |

코드 차이는 다음이다.

```python
# Fine-Tuning
for param in model.parameters():
    param.requires_grad = True

# Transfer Learning
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 10)
```

> 핵심:  
> 파인튜닝은 더 많이 바꾸고, 전이학습은 기존 지식을 최대한 보존한다.

## 6. AdaptiveAvgPool2d 이해하기

사전학습 모델에는 입력 크기가 조금 달라도 마지막 분류기 앞의 feature 크기를 일정하게 맞추는 구조가 자주 들어간다.

### 함수 사용법

```python
nn.AdaptiveAvgPool2d((1, 1))
```

- 입력 feature map의 H, W가 얼마든 출력 공간 크기를 `(1, 1)`로 만든다.
- 입력이 `[N, C, H, W]`이면 출력은 `[N, C, 1, 1]`이 된다.
- 그 다음 `view(N, -1)`로 `[N, C]` 형태로 펼쳐 Linear layer에 넣을 수 있다.

In [ ]:
p = nn.AdaptiveAvgPool2d((1, 1))
l1 = nn.Linear(32, 10)

inputs = torch.randn(100, 32, 16, 16)

m1 = p(inputs)
m2 = m1.view(m1.shape[0], -1)
m3 = l1(m2)

print("inputs:", inputs.shape)
print("after AdaptiveAvgPool2d:", m1.shape)
print("after view:", m2.shape)
print("after Linear:", m3.shape)

코드 해석:

```text
[100, 32, 16, 16]
→ AdaptiveAvgPool2d((1,1))
→ [100, 32, 1, 1]
→ view
→ [100, 32]
→ Linear(32, 10)
→ [100, 10]
```

`AdaptiveAvgPool2d`는 입력 이미지 크기가 달라도 classifier 앞의 feature 차원을 일정하게 맞춰주는 역할을 한다.

## 7. CIFAR-10 클래스와 이미지 전처리

CIFAR-10은 10개 class를 가진다.

```text
plane, car, bird, cat, deer, dog, frog, horse, ship, truck
```

사전학습 모델은 보통 ImageNet 기준 전처리를 사용한다.

### 함수 사용법

```python
transforms.Resize(112)
```

- CIFAR-10의 32×32 이미지를 112×112로 키운다.
- 원래 사전학습 모델은 224×224를 많이 사용하지만, 실습 시간과 메모리 절약을 위해 112를 쓴다.

```python
transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
```

- ImageNet 사전학습 모델에서 자주 쓰는 RGB 평균과 표준편차다.
- 사전학습 가중치를 잘 활용하려면 이 정규화를 맞추는 것이 좋다.

In [ ]:
classes = (
    "plane", "car", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

n_output = len(classes)

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize(112),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0)
])

transform_test = transforms.Compose([
    transforms.Resize(112),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

print("class 수:", n_output)
print(transform_train)

## 8. CIFAR-10 데이터 다운로드와 DataLoader

### 함수 사용법

```python
torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
```

- `root`: 데이터 저장 위치다.
- `train=True`: 학습 데이터다.
- `train=False`: 테스트 데이터다.
- `download=True`: 데이터가 없으면 인터넷에서 다운로드한다.
- `transform`: 이미지에 적용할 전처리다.

```python
DataLoader(dataset, batch_size=50, shuffle=True)
```

- 데이터를 mini-batch로 묶어 공급한다.
- 학습 데이터는 `shuffle=True`로 섞는다.

In [ ]:
data_root = "./data"

train_set_full = torchvision.datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transform_train
)

test_set_full = torchvision.datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transform_test
)

# 빠른 실습용 subset이다. 전체 학습을 원하면 이 부분을 제거하면 된다.
train_size = 3000
test_size = 800

train_set = Subset(train_set_full, list(range(train_size)))
test_set = Subset(test_set_full, list(range(test_size)))

batch_size = 50

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0)

print("train data:", len(train_set))
print("test data:", len(test_set))
print("train batch:", len(train_loader))

## 9. 이미지 미리 보기 함수

정규화된 이미지를 사람이 보기 좋게 다시 복원한다.

### 함수 사용법

```python
image = image * std + mean
image = image.permute(1, 2, 0)
```

- 정규화 해제는 `x = x_norm * std + mean` 구조다.
- PyTorch 이미지는 CHW 순서이므로 Matplotlib에 넣기 위해 HWC로 바꾼다.

In [ ]:
def denormalize_image(image, mean=imagenet_mean, std=imagenet_std):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)

    image = image.cpu() * std_t + mean_t
    image = torch.clamp(image, 0, 1)

    return image


def show_sample_images(loader, classes, num_images=8):
    images, labels = next(iter(loader))

    plt.figure(figsize=(12, 6))

    for i in range(num_images):
        plt.subplot(2, 4, i + 1)

        img = denormalize_image(images[i]).permute(1, 2, 0).numpy()

        plt.imshow(img)
        plt.title(classes[labels[i].item()])
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_sample_images(train_loader, classes, num_images=8)

## 10. ResNet18 사전학습 모델 불러오기

ResNet18은 이미지 분류에서 매우 유명한 사전학습 모델이다.

### 함수 사용법

최신 torchvision 방식은 다음이다.

```python
weights = models.ResNet18_Weights.DEFAULT
net = models.resnet18(weights=weights)
```

예전 노트북에서는 다음처럼 썼다.

```python
net = models.resnet18(pretrained=True)
```

둘 다 의미는 ImageNet으로 학습된 가중치를 불러온다는 것이다.

In [ ]:
def load_resnet18_pretrained():
    try:
        weights = models.ResNet18_Weights.DEFAULT
        net = models.resnet18(weights=weights)
        print("ResNet18 loaded with weights API")
    except Exception:
        net = models.resnet18(pretrained=True)
        print("ResNet18 loaded with pretrained=True")
    return net

net_resnet = load_resnet18_pretrained()

print(net_resnet.fc)
print("fc in_features:", net_resnet.fc.in_features)
print("fc out_features:", net_resnet.fc.out_features)

## 11. ResNet18 마지막 fc layer 교체

ResNet18은 ImageNet 1000개 class를 분류하도록 마지막 layer가 만들어져 있다.

CIFAR-10은 class가 10개이므로 마지막 layer를 교체해야 한다.

### 함수 사용법

```python
fc_in_features = net.fc.in_features
net.fc = nn.Linear(fc_in_features, n_output)
```

- `net.fc.in_features`: 기존 fc layer의 입력 feature 수다.
- `nn.Linear(fc_in_features, 10)`: CIFAR-10에 맞는 새 분류기다.

In [ ]:
torch_seed()

net_resnet = load_resnet18_pretrained()

fc_in_features = net_resnet.fc.in_features
net_resnet.fc = nn.Linear(fc_in_features, n_output)

net_resnet = net_resnet.to(device)

print("교체된 fc:")
print(net_resnet.fc)

## 12. 학습/평가 함수 만들기

원본 노트북에서는 공통 함수 라이브러리를 불러와 `fit`, `evaluate_history`, `show_images_labels`를 사용한다.  
여기서는 바로 실행 가능하도록 필요한 함수를 직접 만든다.

In [ ]:
def fit(model, train_loader, test_loader, criterion, optimizer, device, num_epochs=1):
    history = []

    for epoch in range(num_epochs):
        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            pred = torch.max(outputs, 1)[1]

            train_loss += loss.item() * labels.size(0)
            train_correct += (pred == labels).sum().item()
            train_total += labels.size(0)

        model.eval()

        test_loss = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                pred = torch.max(outputs, 1)[1]

                test_loss += loss.item() * labels.size(0)
                test_correct += (pred == labels).sum().item()
                test_total += labels.size(0)

        history.append([
            epoch + 1,
            train_loss / train_total,
            train_correct / train_total,
            test_loss / test_total,
            test_correct / test_total
        ])

        print(
            f"epoch {epoch + 1} | "
            f"train_loss={history[-1][1]:.4f} | train_acc={history[-1][2]:.4f} | "
            f"test_loss={history[-1][3]:.4f} | test_acc={history[-1][4]:.4f}"
        )

    return np.array(history)


def evaluate_history(history, title="Learning Curve"):
    plt.plot(history[:, 0], history[:, 1], label="train loss")
    plt.plot(history[:, 0], history[:, 3], label="test loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title + " Loss")
    plt.legend()
    plt.show()

    plt.plot(history[:, 0], history[:, 2], label="train acc")
    plt.plot(history[:, 0], history[:, 4], label="test acc")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.title(title + " Accuracy")
    plt.legend()
    plt.show()

    print("최종 test accuracy:", history[-1, 4])

In [ ]:
def show_predictions(model, loader, classes, device, num_images=12):
    model.eval()

    images, labels = next(iter(loader))
    images_device = images.to(device)

    with torch.no_grad():
        outputs = model(images_device)
        predicted = torch.max(outputs, 1)[1].cpu()

    plt.figure(figsize=(14, 8))

    for i in range(num_images):
        plt.subplot(3, 4, i + 1)

        img = denormalize_image(images[i]).permute(1, 2, 0).numpy()

        true_label = classes[labels[i].item()]
        pred_label = classes[predicted[i].item()]

        color = "green" if labels[i].item() == predicted[i].item() else "red"

        plt.imshow(img)
        plt.title(f"정답: {true_label}\n예측: {pred_label}", color=color, fontsize=10)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 13. ResNet18 파인튜닝 학습

파인튜닝에서는 사전학습 모델 전체를 새 데이터에 맞게 조금씩 조정한다.

### 학습 설정

```python
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
```

- `CrossEntropyLoss`: 다중 분류 손실함수다.
- `SGD + Momentum`: 강의에서 파인튜닝에 안정적이라고 설명한 조합이다.
- `lr=0.001`: 사전학습 가중치를 너무 크게 흔들지 않도록 작은 학습률을 쓴다.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    net_resnet.parameters(),
    lr=0.001,
    momentum=0.9
)

num_epochs = 1

history_resnet = fit(
    net_resnet,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    device,
    num_epochs=num_epochs
)

evaluate_history(history_resnet, title="ResNet18 Fine-Tuning")

In [ ]:
show_predictions(net_resnet, test_loader, classes, device, num_images=12)

## 14. Transfer Learning: feature extractor 고정하기

데이터가 적을 때는 모델 전체를 학습시키지 않고 앞쪽 feature extractor를 고정할 수 있다.

### 함수 사용법

```python
for param in net.parameters():
    param.requires_grad = False
```

- 해당 parameter는 gradient 계산과 업데이트 대상에서 빠진다.
- 마지막 layer만 새로 만들면 마지막 layer만 학습된다.

In [ ]:
torch_seed()

net_transfer = load_resnet18_pretrained()

for param in net_transfer.parameters():
    param.requires_grad = False

fc_in_features = net_transfer.fc.in_features
net_transfer.fc = nn.Linear(fc_in_features, n_output)

net_transfer = net_transfer.to(device)

trainable_params = sum(p.numel() for p in net_transfer.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in net_transfer.parameters())

print("total params:", total_params)
print("trainable params:", trainable_params)
print("교체된 fc만 학습 대상이다:")
print(net_transfer.fc)

> 기억할 점:  
> `requires_grad=False`를 먼저 적용하고 마지막 layer를 새로 교체하면, 새 layer는 기본적으로 `requires_grad=True`라서 학습 대상이 된다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net_transfer.fc.parameters(), lr=0.001, momentum=0.9)

history_transfer = fit(
    net_transfer,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    device,
    num_epochs=1
)

evaluate_history(history_transfer, title="ResNet18 Transfer Learning")

## 15. VGG19-BN 모델 구조와 classifier[6] 교체

VGG19-BN은 ResNet18과 구조가 다르다.

ResNet18은 마지막 layer가 `net.fc`였지만, VGG19-BN은 마지막 선형 layer가 다음 위치에 있다.

```python
net.classifier[6]
```

### 함수 사용법

```python
in_features = net.classifier[6].in_features
net.classifier[6] = nn.Linear(in_features, n_output)
```

> 주의:  
> VGG19-BN은 모델이 크고 다운로드도 무겁다.  
> 아래 코드는 기본적으로 구조 확인만 하도록 `RUN_VGG = False`로 두었다.

In [ ]:
RUN_VGG = False

if RUN_VGG:
    try:
        weights = models.VGG19_BN_Weights.DEFAULT
        net_vgg = models.vgg19_bn(weights=weights)
    except Exception:
        net_vgg = models.vgg19_bn(pretrained=True)

    print("기존 마지막 layer:")
    print(net_vgg.classifier[6])

    in_features = net_vgg.classifier[6].in_features
    net_vgg.classifier[6] = nn.Linear(in_features, n_output)

    # 112x112 입력 실습을 위해 원본 노트북처럼 마지막 pooling과 avgpool을 조정할 수 있다.
    net_vgg.features = net_vgg.features[:-1]
    net_vgg.avgpool = nn.Identity()

    net_vgg = net_vgg.to(device)

    print("교체 후 마지막 layer:")
    print(net_vgg.classifier[6])
else:
    print("VGG19-BN은 무거워서 기본 실행에서는 건너뛴다.")
    print("실행하려면 RUN_VGG = True로 바꾼다.")

## 16. MobileNetV2 초급 이미지 분류기

강의의 초급 이미지 분류기 실습에서는 MobileNetV2를 사용한다.

MobileNetV2는 비교적 가볍고 빠른 사전학습 모델이다.  
모바일 환경이나 가벼운 실습에 적합하다.

마지막 layer 위치는 다음이다.

```python
model.classifier[1]
```

기존 ImageNet용 출력은 1000개 class다.  
CIFAR-10에 맞게 10개 class로 바꾼다.

In [ ]:
def load_mobilenet_v2_pretrained():
    try:
        weights = models.MobileNet_V2_Weights.DEFAULT
        model = models.mobilenet_v2(weights=weights)
        print("MobileNetV2 loaded with weights API")
    except Exception:
        model = models.mobilenet_v2(pretrained=True)
        print("MobileNetV2 loaded with pretrained=True")
    return model

model_mobile = load_mobilenet_v2_pretrained()

print("기존 classifier:")
print(model_mobile.classifier)

num_features = model_mobile.classifier[1].in_features
model_mobile.classifier[1] = nn.Linear(num_features, n_output)

model_mobile = model_mobile.to(device)

print("\n교체 후 classifier:")
print(model_mobile.classifier)

## 17. MobileNetV2 학습 설정과 짧은 학습

초급 실습의 기본 설정은 다음과 같다.

```python
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
num_epochs = 20
```

여기서는 실행 시간을 줄이기 위해 1 epoch만 기본으로 둔다.  
실제 성능을 보고 싶으면 `num_epochs`를 늘리면 된다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_mobile.parameters(), lr=0.001, momentum=0.9)

history_mobile = fit(
    model_mobile,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    device,
    num_epochs=1
)

evaluate_history(history_mobile, title="MobileNetV2 Fine-Tuning")

## 18. 클래스별 정확도 계산

모델이 전체적으로 몇 퍼센트를 맞히는지도 중요하지만, 어떤 class를 잘 맞히고 어떤 class를 어려워하는지도 봐야 한다.

### 함수 사용법

```python
evaluate_per_class(model, loader, classes, device)
```

- class별 맞힌 개수와 전체 개수를 따로 센다.
- 가장 잘 맞추는 class와 가장 어려워하는 class를 확인한다.

In [ ]:
def evaluate_per_class(model, loader, classes, device):
    model.eval()

    class_correct = [0] * len(classes)
    class_total = [0] * len(classes)

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            predicted = torch.max(outputs, 1)[1]

            correct = (predicted == labels)

            for i in range(len(labels)):
                label = labels[i].item()
                class_correct[label] += correct[i].item()
                class_total[label] += 1

    accuracies = []

    print("클래스별 정확도")
    print("=" * 50)

    for i, class_name in enumerate(classes):
        if class_total[i] > 0:
            acc = 100 * class_correct[i] / class_total[i]
        else:
            acc = 0.0

        accuracies.append(acc)
        print(f"{class_name:8s}: {acc:5.1f}%")

    best_idx = int(np.argmax(accuracies))
    worst_idx = int(np.argmin(accuracies))

    print("=" * 50)
    print(f"평균 정확도: {np.mean(accuracies):.2f}%")
    print(f"가장 잘 맞추는 것: {classes[best_idx]} ({accuracies[best_idx]:.1f}%)")
    print(f"가장 어려워하는 것: {classes[worst_idx]} ({accuracies[worst_idx]:.1f}%)")

evaluate_per_class(model_mobile, test_loader, classes, device)

## 19. 모델 저장하기

학습한 모델의 parameter를 저장할 수 있다.

### 함수 사용법

```python
torch.save(model.state_dict(), model_path)
```

- 모델 구조 자체가 아니라 학습된 가중치 딕셔너리를 저장한다.
- 나중에 같은 구조의 모델을 만든 뒤 `load_state_dict()`로 불러온다.

In [ ]:
model_path = "/mnt/data/day13_mobilenetv2_cifar10_state_dict.pth"

torch.save(model_mobile.state_dict(), model_path)

print("모델 저장 완료:", model_path)
print("나중에 불러오는 기본 흐름:")
print("model = models.mobilenet_v2(weights=None)")
print("model.classifier[1] = nn.Linear(1280, 10)")
print(f"model.load_state_dict(torch.load('{model_path}', map_location='cpu'))")
print("model.eval()")

## 20. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `pretrained` | 사전학습된 | 이미 학습된 가중치 사용 |
| `weights` | 사전학습 가중치 | `models.ResNet18_Weights.DEFAULT` |
| `ImageNet` | 대규모 이미지 데이터셋 | 사전학습에 자주 사용 |
| `fine-tuning` | 파인튜닝 | 전체 또는 많은 파라미터 재학습 |
| `transfer learning` | 전이학습 | 일부를 고정하고 마지막 layer 중심 학습 |
| `requires_grad` | gradient 계산 여부 | `False`면 학습에서 제외 |
| `AdaptiveAvgPool2d` | 원하는 출력 크기 풀링 | `nn.AdaptiveAvgPool2d((1,1))` |
| `ResNet18` | 잔차 연결 기반 CNN | `models.resnet18(...)` |
| `VGG19-BN` | VGG19 + BatchNorm | `models.vgg19_bn(...)` |
| `MobileNetV2` | 가벼운 CNN 모델 | `models.mobilenet_v2(...)` |
| `fc` | fully connected layer | ResNet의 마지막 분류기 |
| `classifier[6]` | VGG의 마지막 Linear | 1000 class를 10 class로 교체 |
| `classifier[1]` | MobileNetV2의 마지막 Linear | 1000 class를 10 class로 교체 |
| `in_features` | Linear 입력 feature 수 | 새 layer 만들 때 사용 |
| `out_features` | Linear 출력 class 수 | CIFAR-10은 10 |
| `state_dict` | 모델 파라미터 딕셔너리 | 저장/불러오기 |

## 21. 시험용 요약

```text
사전학습 모델 = 큰 데이터셋으로 이미 학습된 모델을 가져와 내 문제에 맞게 바꾸는 방식
```

꼭 기억할 것:

- 사전학습 모델은 ImageNet 같은 대규모 데이터로 이미 학습된 모델이다.
- 처음부터 학습하는 것보다 빠르고 성능이 좋을 수 있다.
- Fine-Tuning은 모델 전체 또는 많은 부분을 새 데이터로 다시 조정한다.
- Transfer Learning은 feature extractor를 고정하고 마지막 분류기 위주로 학습한다.
- `requires_grad=False`는 해당 파라미터를 학습하지 않겠다는 뜻이다.
- `AdaptiveAvgPool2d((1,1))`은 입력 크기와 무관하게 `[N,C,1,1]`을 만든다.
- CIFAR-10은 10개 class다.
- 사전학습 모델을 쓸 때는 ImageNet mean/std 정규화를 맞추는 것이 좋다.
- ResNet18의 마지막 layer는 `net.fc`다.
- ResNet18은 `net.fc = nn.Linear(net.fc.in_features, 10)`으로 교체한다.
- VGG19-BN의 마지막 layer는 `net.classifier[6]`이다.
- MobileNetV2의 마지막 layer는 `model.classifier[1]`이다.
- `CrossEntropyLoss`는 다중 분류 손실함수다.
- 파인튜닝에는 작은 learning rate와 SGD with Momentum을 많이 사용한다.
- 모델 저장은 `torch.save(model.state_dict(), path)`로 한다.